In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications
!pip install -e ".[experiment]"

In [ ]:
!pip install .

In [ ]:
!pip install -q "peft==0.13.2" "transformers==4.41.0"

In [ ]:
import os

BASE_RESULTS_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments"
LOG_DIR = os.path.join(BASE_RESULTS_DIR, "logs")
MODEL_DIR = os.path.join(BASE_RESULTS_DIR, "models")
BASE_EVAL_DIR = os.path.join(BASE_RESULTS_DIR, "eval")

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(BASE_EVAL_DIR, exist_ok=True)

In [ ]:
import logging
import sys

# Get the root logger or a named logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)  # allow INFO and above

for h in list(logger.handlers):
  logger.removeHandler(h)

# Create a handler that writes to stdout
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)

# (Optional) set a formatting for readability
formatter = logging.Formatter('%(levelname)s - %(message)s')
handler.setFormatter(formatter)

# Add handler to the logger
logger.addHandler(handler)

# Now test
logger.info("This will be printed to stdout")
logger.debug("This will not print (level is INFO)")

INFO - This will be printed to stdout


In [ ]:
import os
import json
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate
from tqdm import tqdm

# ----------------------------
# 1. Config
# ----------------------------
MODEL_NAME = "google/flan-t5-base"   # or "google/flan-t5-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 16                 # changed (pick what you want)
LIMIT = None                         # number of validation examples

SAVE_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/"
os.makedirs(SAVE_DIR, exist_ok=True)
OUT_PATH = os.path.join(SAVE_DIR, "results_flant5_squadv1.json")

print("Device:", DEVICE)

# ----------------------------
# 2. Load model & tokenizer
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# For T5/FLAN-T5, pad_token_id is usually defined; keep this safeguard anyway
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ----------------------------
# 3. Load SQuAD1.1 validation
# ----------------------------
dataset = load_dataset("squad", split="validation")
if LIMIT is not None:
    dataset = dataset.select(range(LIMIT))

print("Evaluating on", len(dataset), "examples")

# ----------------------------
# 4. Run generation
# ----------------------------
predictions = []
references = []

for doc in tqdm(dataset):
    prompt = (
        "Answer the question using the context.\n"
        "Context: " + doc["context"]
        + "\nQuestion: " + doc["question"]
        + "\nAnswer:"
    )

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,   # safer for T5
        padding=False,
    ).to(DEVICE)

    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=4,  # helps quality for QA
            early_stopping=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    # For seq2seq models, out is the generated sequence (no need to slice input tokens)
    text = tokenizer.decode(out[0], skip_special_tokens=True)

    # Stop at first newline and FORCE the output to end with '\n'
    text = text.split("\n", 1)[0].strip() + "\n"

    # DEBUG: print first few
    if len(predictions) < 5:
        print("==== DEBUG EXAMPLE ====")
        print("Q   :", doc["question"])
        print("GT  :", doc["answers"]["text"])
        print("PRED:", repr(text))

    predictions.append({
        "id": doc["id"],
        "prediction_text": text.strip(),  # SQuAD metric expects text without trailing newline
    })
    references.append({
        "id": doc["id"],
        "answers": doc["answers"],
    })

# ----------------------------
# 5. Compute SQuAD1 metrics
# ----------------------------
metric = evaluate.load("squad")  # SQuAD 1.1 EM/F1
scores = metric.compute(predictions=predictions, references=references)
print("Scores:", scores)

# ----------------------------
# 6. Save results
# ----------------------------
with open(OUT_PATH, "w") as f:
    json.dump({
        "results": scores,
        "num_examples": len(dataset),
        "model": MODEL_NAME,
        "max_new_tokens": MAX_NEW_TOKENS,
    }, f, indent=2)

print("Saved to:", OUT_PATH)


Device: cuda
Evaluating on 10570 examples


  0%|          | 1/10570 [00:00<32:02,  5.50it/s]

==== DEBUG EXAMPLE ====
Q   : Which NFL team represented the AFC at Super Bowl 50?
GT  : ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']
PRED: 'Denver Broncos\n'


  0%|          | 2/10570 [00:00<27:18,  6.45it/s]

==== DEBUG EXAMPLE ====
Q   : Which NFL team represented the NFC at Super Bowl 50?
GT  : ['Carolina Panthers', 'Carolina Panthers', 'Carolina Panthers']
PRED: 'Carolina Panthers\n'


...

==== DEBUG EXAMPLE ====
Q   : Where did Super Bowl 50 take place?
GT  : ['Santa Clara, California', "Levi's Stadium", "Levi's Stadium in the San Francisco Bay Area at Santa Clara, California."]
PRED: 'Santa Clara, California\n'
==== DEBUG EXAMPLE ====
Q   : Which NFL team won Super Bowl 50?
GT  : ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']
PRED: 'Denver Broncos\n'


  0%|          | 6/10570 [00:00<24:58,  7.05it/s]

==== DEBUG EXAMPLE ====
Q   : What color was used to emphasize the 50th anniversary of the Super Bowl?
GT  : ['gold', 'gold', 'gold']
PRED: 'gold\n'


100%|██████████| 10570/10570 [27:09<00:00,  6.49it/s]


Scores: {'exact_match': 77.67265846736045, 'f1': 87.43215729970268}
Saved to: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/results_flant5_squadv1.json


In [ ]:
#squad1 with config batch_size
import os
import json
import math
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate
from tqdm import tqdm

# ----------------------------
# 1. Config
# ----------------------------
MODEL_NAME = "google/flan-t5-base"   # or "google/flan-t5-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 16
LIMIT = None                         # None = full validation
BATCH_SIZE = 8                       # <-- configurable batch size

SAVE_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/"
os.makedirs(SAVE_DIR, exist_ok=True)
OUT_PATH = os.path.join(SAVE_DIR, "results_flant5_squadv1_batched.json")

print("Device:", DEVICE)
print("Batch size:", BATCH_SIZE)

# ----------------------------
# 2. Load model & tokenizer
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ----------------------------
# 3. Load SQuAD1.1 validation
# ----------------------------
dataset = load_dataset("squad", split="validation")
if LIMIT is not None:
    dataset = dataset.select(range(LIMIT))

print("Evaluating on", len(dataset), "examples")

# ----------------------------
# 4. Run generation (batched)
# ----------------------------
predictions = []
references = []

def build_prompt(doc):
    return (
        "Answer the question using the context.\n"
        "Context: " + doc["context"]
        + "\nQuestion: " + doc["question"]
        + "\nAnswer:"
    )

num_batches = math.ceil(len(dataset) / BATCH_SIZE)

for b in tqdm(range(num_batches), desc="Batches"):
    start = b * BATCH_SIZE
    end = min((b + 1) * BATCH_SIZE, len(dataset))
    batch = dataset.select(range(start, end))

    prompts = [build_prompt(d) for d in batch]

    enc = tokenizer(
        prompts,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True,  # <-- needed for batching
    ).to(DEVICE)

    with torch.no_grad():
        outs = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=4,
            early_stopping=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    decoded = tokenizer.batch_decode(outs, skip_special_tokens=True)

    for i, doc in enumerate(batch):
        # Stop at first newline (like before); we keep newline in debug only if you want
        text = decoded[i].split("\n", 1)[0].strip()

        # DEBUG: print first few
        if len(predictions) < 5:
            print("==== DEBUG EXAMPLE ====")
            print("Q   :", doc["question"])
            print("GT  :", doc["answers"]["text"])
            print("PRED:", repr(text))

        predictions.append({
            "id": doc["id"],
            "prediction_text": text,
        })
        references.append({
            "id": doc["id"],
            "answers": doc["answers"],
        })

# ----------------------------
# 5. Compute SQuAD1 metrics
# ----------------------------
metric = evaluate.load("squad")
scores = metric.compute(predictions=predictions, references=references)
print("Scores:", scores)

# ----------------------------
# 6. Save results
# ----------------------------
with open(OUT_PATH, "w") as f:
    json.dump({
        "results": scores,
        "num_examples": len(dataset),
        "model": MODEL_NAME,
        "max_new_tokens": MAX_NEW_TOKENS,
        "batch_size": BATCH_SIZE,
    }, f, indent=2)

print("Saved to:", OUT_PATH)

In [ ]:
#now i wanna do some samples test on squad2 always using FLANT5

In [ ]:
# ==== Colab: LM-Eval (Python API) on SQuADv2 with FLAN-T5 (clean outputs) ====

import os
import json
import torch
import lm_eval
from lm_eval.models.huggingface import HFLM

# ----------------------------
# Config
# ----------------------------
MODEL_NAME  = "google/flan-t5-large"  # or flan-t5-xl on A100 if you want
TASKS       = ["squadv2"]
BATCH_SIZE  = 8
LIMIT       = 20          # None = full validation
NUM_FEWSHOT = 0
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

SAVE_DIR = "lmeval_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)
OUT_FILE = os.path.join(SAVE_DIR, f"metrics_{MODEL_NAME.replace('/','_')}_{'_'.join(TASKS)}.json")

# ----------------------------
# Build LM-Eval model wrapper
# ----------------------------
hflm = HFLM(
    pretrained=MODEL_NAME,
    tokenizer=MODEL_NAME,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    dtype="float16" if DEVICE.startswith("cuda") else "float32",
)

# ----------------------------
# Run evaluation (no sample logging, no write_out)
# ----------------------------
res = lm_eval.simple_evaluate(
    model=hflm,
    tasks=TASKS,
    num_fewshot=NUM_FEWSHOT,
    batch_size=BATCH_SIZE,
    limit=LIMIT,
    write_out=False,
    log_samples=False,
)

# ----------------------------
# Extract + print only the metrics we care about
# ----------------------------


print("📊 SQuAD v2 Results")
print("------------------")
print(f"Results: {res["results"]}")

In [ ]:
#better code for squadv2 dense model, but also the previous one works

In [ ]:
import json

import lm_eval
# import torch
from lm_eval import tasks
from lm_eval import utils as lm_eval_utils
from lm_eval.api.registry import ALL_TASKS
from lm_eval.models.huggingface import HFLM
from lm_eval.tasks import initialize_tasks

from slicegpt import gpu_utils, hf_utils, utils
from slicegpt.config import config

logging.basicConfig(
    level=logging.DEBUG,
    stream=sys.stdout,
    format="%(levelname)s: %(message)s"
)
def eval(args):
    """
    Run LM Evaluation Harness on either:
      - a sliced (pruned) model, if 'sliced_model_path' is provided
      - or a dense HF model, if not.
    """

    logger.info("Running Evaluation")

    use_sliced = args.get("sliced_model_path") is not None and args.get("sparsity", 0.0) > 0.0

    if use_sliced:
        # ---------- SLICED MODEL BRANCH ----------
        logger.info(
            f"Loading SLICED model {args['model']} from {args['sliced_model_path']} "
            f"with sparsity {args['sparsity']}"
        )

        model_adapter, tokenizer = hf_utils.load_sliced_model(
            args["model"],
            args["sliced_model_path"],
            sparsity=args["sparsity"],
            token=None,
            round_interval=args.get("round_interval", None),
        )

        # For sliced models, disable weight tying
        if hasattr(model_adapter.model, "tie_weights"):
            model_adapter.model.tie_weights = lambda *x, **kw: None

        model_adapter.model.to(config.device)

        hflm = HFLM(
            pretrained=model_adapter.model,
            tokenizer=tokenizer,
            batch_size=args["batch_size"],
        )

    else:
        # ---------- DENSE BASELINE BRANCH ----------
        logger.info(
            f"Loading DENSE baseline model {args['model']} directly from HF hub"
        )

        # Here we let LM Eval Harness load the model & tokenizer itself
        hflm = HFLM(
            pretrained=args["model"],
            batch_size=args["batch_size"],
        )

    # ---------- Task selection ----------
    if args["tasks"] is None:
        logger.warning(
            "args['tasks'] is None -> using ALL_TASKS. "
            "This may be very slow / memory-heavy."
        )
        task_names = tasks.ALL_TASKS
    else:
        if isinstance(args["tasks"], str):
            patterns = [t.strip() for t in args["tasks"].split(",") if t.strip()]
        else:
            patterns = args["tasks"]
        task_names = lm_eval_utils.pattern_match(patterns, ALL_TASKS)

    logger.info(f"Selected Tasks: {task_names}")

    # ---------- Run evaluation ----------
    results = lm_eval.simple_evaluate(
        hflm,
        tasks=task_names,
        num_fewshot=args["num_fewshot"],
        batch_size=args["batch_size"],
        limit=args["limit"],
        write_out=True,
        log_samples=False,
    )

    logger.info("Results (metrics only):")
    logger.info(results["results"])


    # ---------- Save results ----------
    os.makedirs(args["save_dir"], exist_ok=True)

    # Use sparsity=0.0 if not present, so filenames still make sense
    sparsity_val = float(args.get("sparsity", 0.0))
    sparsity_tag = f"{sparsity_val:.2f}"

    result_path = os.path.join(
        args["save_dir"],
        f"results_s{sparsity_tag}_{'_'.join(task_names)}_5shots.json",
    )

    with open(result_path, "w") as f:
        json.dump(results, f, indent=2)

    logger.info(f"Saved results to {result_path}")

    return results


In [ ]:
args = {
    "model": "google/flan-t5-base", #to use with FLANT5
    "sliced_model_path": None,   # IMPORTANT
    "sparsity": 0.0,
    "round_interval": None,
    "batch_size": 16,
    "tasks": ["squadv2"],
    "num_fewshot": None,
    "limit": 20,
    "save_dir": "/content/",
}

eval(args)

INFO - Running Evaluation
INFO - Loading DENSE baseline model google/flan-t5-base directly from HF hub
INFO - Using device 'cuda'


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


INFO - Selected Tasks: ['squadv2']


TypeError: must be called with a dataclass type or instance

In [ ]:
args = {
    "model": "facebook/opt-125m",
    #"model": "facebook/opt-1.3b",
    "sliced_model_path": None,   # IMPORTANT
    "sparsity": 0.0,
    "round_interval": None,
    "batch_size": 16,
    "tasks": ["squadv2"],
    "num_fewshot": 2,
    "limit": None,
    "save_dir": "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/eval/squad_dense_only/",
}

eval(args)

INFO - Running Evaluation
INFO - Loading DENSE baseline model facebook/opt-125m directly from HF hub
INFO - Using device 'cuda'
INFO - Selected Tasks: ['squadv2']


TypeError: must be called with a dataclass type or instance

In [ ]:
#rn the previous cell is not working, i think there is some library conflicts